# Fase 2c: Segmentación Conductual (Clustering)

**Scope:** Segmentar clientes en 4-6 clusters conductuales con centroides fijos para comparabilidad temporal.

**8 features de clustering:**
1. `frequency_monthly_avg` — frecuencia mensual de compras
2. `monetary_monthly_avg` — gasto mensual promedio
3. `redeem_rate` — tasa de canje de puntos
4. `retailer_entropy` — diversidad de retailers (Shannon entropy)
5. `pct_redeem_digital` — % canjes digitales (proxy de pct_digital)
6. `earn_velocity_90` — puntos ganados en 90d
7. `days_since_last_activity` — recencia
8. `points_pressure` — presión por vencimiento de puntos

**Arquetipos esperados:** Dormidos, Exploradores, Cazadores de Canje, Heavy Users, Digitales, En Riesgo

**Centroides fijos:** Se entrenan en un t0, se aplican a todos los demás para tracking temporal.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, adjusted_rand_score

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')

print("Imports OK")

In [ ]:
# ── Carga de datos ─────────────────────────────────────────────────────────
USE_MOCK = True

if USE_MOCK:
    import duckdb
    with open("../../fase1/test_mock_local.py") as f:
        code = f.read().replace("con.close()", "# con.close()")
    _globals = {}
    exec(code, _globals)
    con = _globals['con']
    df = con.execute("SELECT * FROM customer_snapshot").df()
    con.close()
    print(f"Datos cargados desde mock DuckDB: {df.shape}")
else:
    from google.cloud import bigquery
    client = bigquery.Client(project="my-gcp-project")
    df = client.query("SELECT * FROM `my-gcp-project.loyalty_intelligence.customer_snapshot`").to_dataframe()
    print(f"Datos cargados desde BigQuery: {df.shape}")

df['t0'] = pd.to_datetime(df['t0'])

# Calcular retailer_entropy basado en frecuencia (no spend) para alinear con features conductuales
if 'retailer_entropy' not in df.columns:
    freqs = df[['freq_store_a', 'freq_store_b', 'freq_store_c', 'freq_store_d', 'freq_store_e']].values
    tot = np.where(freqs.sum(axis=1, keepdims=True) == 0, 1, freqs.sum(axis=1, keepdims=True))
    p = freqs / tot
    df['retailer_entropy'] = -(p * np.where(p > 0, np.log(p), 0)).sum(axis=1)

print(f"Shape: {df.shape}, t0s: {sorted(df['t0'].dt.strftime('%Y-%m').unique())}")

## 1. Preparar Features de Clustering

In [ ]:
# ── 8 features de clustering ─────────────────────────────────────
CLUSTER_FEATURES = [
    'frequency_monthly_avg',
    'monetary_monthly_avg',
    'redeem_rate',
    'retailer_entropy',        # proxy de retailer_diversity
    'pct_redeem_digital',      # proxy de pct_digital
    'earn_velocity_90',
    'days_since_last_activity',
    'points_pressure',
]

# Verificar disponibilidad
missing = [f for f in CLUSTER_FEATURES if f not in df.columns]
if missing:
    print(f"WARN: Features faltantes: {missing}")
    CLUSTER_FEATURES = [f for f in CLUSTER_FEATURES if f in df.columns]

# ── fillna CONTEXTUAL por feature (no blanket 0) ────────────────
df_clust = df[['cust_id', 't0', 'y'] + CLUSTER_FEATURES].copy()

# days_since_last_activity: NaN = nunca activo → valor alto
df_clust['days_since_last_activity'] = df_clust['days_since_last_activity'].fillna(999)

# pct_redeem_digital: NaN = nunca canjeó → 0
df_clust['pct_redeem_digital'] = df_clust['pct_redeem_digital'].fillna(0)

# redeem_rate: NaN = sin canjes → 0
df_clust['redeem_rate'] = df_clust['redeem_rate'].fillna(0)

# Resto: 0 como default razonable
df_clust[CLUSTER_FEATURES] = df_clust[CLUSTER_FEATURES].fillna(0)

print(f"Features de clustering: {len(CLUSTER_FEATURES)}")
print(df_clust[CLUSTER_FEATURES].describe().round(2).to_string())

In [ ]:
# ── Diagnostico: correlacion entre features ──────────────────────
# Features correlacionadas reciben doble peso implicito en KMeans
corr = df_clust[CLUSTER_FEATURES].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            mask=mask, ax=ax, vmin=-1, vmax=1)
ax.set_title('Correlacion entre Features de Clustering')
plt.tight_layout()
plt.show()

# Alertar pares con |r| > 0.7
high_corr = []
for i in range(len(CLUSTER_FEATURES)):
    for j in range(i+1, len(CLUSTER_FEATURES)):
        r = corr.iloc[i, j]
        if abs(r) > 0.7:
            high_corr.append((CLUSTER_FEATURES[i], CLUSTER_FEATURES[j], r))

if high_corr:
    print("ALERTA: pares con |r| > 0.7 (considerar eliminar uno):")
    for f1, f2, r in high_corr:
        print(f"  {f1} ↔ {f2}: r={r:.3f}")
else:
    print("OK: ningún par con |r| > 0.7 — features suficientemente independientes")

In [ ]:
# ── Pre-procesamiento: log-transform + winsorize + StandardScaler ─
from scipy.stats import skew

t0_train = df_clust['t0'].min()  # primer t0 = base para centroides
mask_train = df_clust['t0'] == t0_train

# 1) Detectar features con skew > 2 → log1p
skews = df_clust.loc[mask_train, CLUSTER_FEATURES].apply(skew)
LOG_FEATURES = skews[skews.abs() > 2].index.tolist()
print(f"Features con |skew| > 2 (se aplica log1p): {LOG_FEATURES}")
print(f"  Skews: {skews.round(2).to_dict()}")

df_proc = df_clust[CLUSTER_FEATURES].copy()
for feat in LOG_FEATURES:
    df_proc[feat] = np.log1p(df_proc[feat])

# 2) Winsorize en percentil 1-99 (calculado en TRAIN, aplicado a todo)
CLIP_LO = {}
CLIP_HI = {}
for feat in CLUSTER_FEATURES:
    lo = df_proc.loc[mask_train, feat].quantile(0.01)
    hi = df_proc.loc[mask_train, feat].quantile(0.99)
    CLIP_LO[feat] = lo
    CLIP_HI[feat] = hi
    df_proc[feat] = df_proc[feat].clip(lo, hi)

# 3) StandardScaler fit en TRAIN
scaler = StandardScaler()
scaler.fit(df_proc.loc[mask_train])

X_scaled = scaler.transform(df_proc)
X_train_scaled = X_scaled[mask_train.values]

print(f"\nScaler fit en t0={t0_train.strftime('%Y-%m')}: {mask_train.sum()} filas")
print(f"X_scaled shape: {X_scaled.shape}")
print(f"Rango post-transform (train): min={X_train_scaled.min():.2f}, max={X_train_scaled.max():.2f}")

In [ ]:
# ── Distribucion de las 8 features ───────────────────────────────
n_feat = len(CLUSTER_FEATURES)
ncols = 4
nrows = (n_feat + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(16, 3 * nrows))
axes = axes.flatten()

for i, feat in enumerate(CLUSTER_FEATURES):
    df_clust[feat].hist(bins=30, ax=axes[i], color='steelblue', edgecolor='white')
    axes[i].set_title(feat, fontsize=10)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribucion de Features de Clustering', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

## 2. Determinar K Óptimo

In [ ]:
# ── Elbow + Silhouette para k=2..8 ─────────────────────────────
K_RANGE = range(2, 9)
inertias = []
silhouettes = []

for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_train_scaled)
    inertias.append(km.inertia_)
    sil = silhouette_score(X_train_scaled, labels)
    silhouettes.append(sil)
    print(f"  k={k}: inertia={km.inertia_:,.0f}, silhouette={sil:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(list(K_RANGE), inertias, 'bo-', linewidth=2)
axes[0].set_xlabel('K')
axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Method')

axes[1].plot(list(K_RANGE), silhouettes, 'ro-', linewidth=2)
axes[1].set_xlabel('K')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score por K')

plt.tight_layout()
plt.show()

In [ ]:
# ── Elegir K optimo ───────────────────────────────────────────
# Preferir K en rango 4-6 (spec del prompt), con mejor silhouette
preferred = {k: s for k, s in zip(K_RANGE, silhouettes) if 4 <= k <= 6}
K_OPT = max(preferred, key=preferred.get) if preferred else int(list(K_RANGE)[np.argmax(silhouettes)])

print(f"K optimo seleccionado: {K_OPT} (silhouette={preferred.get(K_OPT, silhouettes[K_OPT-2]):.4f})")
print(f"Rango evaluado: {dict(zip(K_RANGE, [f'{s:.4f}' for s in silhouettes]))}")

## 3. KMeans con Centroides Fijos

In [ ]:
# ── Entrenar KMeans en primer t0, aplicar a todos ───────────────
kmeans = KMeans(n_clusters=K_OPT, random_state=42, n_init=10)
kmeans.fit(X_train_scaled)

# Aplicar centroides fijos a TODOS los t0s
df_clust['cluster'] = kmeans.predict(X_scaled)

print(f"KMeans entrenado en t0={t0_train.strftime('%Y-%m')} con K={K_OPT}")
print(f"\nDistribucion global de clusters:")
for c in sorted(df_clust['cluster'].unique()):
    n = (df_clust['cluster'] == c).sum()
    print(f"  Cluster {c}: {n:,} ({n/len(df_clust)*100:.1f}%)")

In [ ]:
# ── Perfiles de clusters (media de features por cluster, solo train) ──────
profiles = df_clust.loc[mask_train].groupby('cluster')[CLUSTER_FEATURES].mean()

# Normalizar para comparacion (0-1 por feature)
profiles_norm = (profiles - profiles.min()) / (profiles.max() - profiles.min() + 1e-9)

print("Perfiles de clusters (valores promedio, train only):")
print(profiles.round(2).to_string())
print("\nPerfiles normalizados (0-1):")
print(profiles_norm.round(2).to_string())

In [ ]:
# ── Asignar nombres con scoring multi-dimensional (Hungarian) ────
from scipy.optimize import linear_sum_assignment

def assign_cluster_names_hungarian(profiles_norm):
    """Asigna nombres a clusters usando scoring por arquetipo + asignacion optima."""
    pn = profiles_norm.copy()
    
    # Definir arquetipos como vectores de peso sobre features
    # Cada arquetipo tiene features que lo definen (positivo = alto, negativo = bajo)
    archetypes = {
        'Dormidos': {
            'days_since_last_activity': 2.0,
            'frequency_monthly_avg': -1.0,
            'monetary_monthly_avg': -1.0,
        },
        'Heavy Users': {
            'frequency_monthly_avg': 2.0,
            'monetary_monthly_avg': 1.5,
            'retailer_entropy': 1.0,
        },
        'Cazadores de Canje': {
            'redeem_rate': 2.5,
            'points_pressure': 1.0,
        },
        'Digitales': {
            'pct_redeem_digital': 2.5,
            'redeem_rate': 0.5,
        },
        'En Riesgo': {
            'points_pressure': 1.5,
            'days_since_last_activity': 1.0,
            'frequency_monthly_avg': -0.5,
        },
        'Exploradores': {
            'retailer_entropy': 1.5,
            'frequency_monthly_avg': 0.5,
            'redeem_rate': -1.0,
        },
    }
    
    cluster_ids = list(pn.index)
    archetype_names = list(archetypes.keys())
    
    # Calcular score de cada cluster para cada arquetipo
    n_clusters = len(cluster_ids)
    n_archetypes = len(archetype_names)
    
    # Matriz de scores (clusters x arquetipos)
    score_matrix = np.zeros((n_clusters, n_archetypes))
    for i, c in enumerate(cluster_ids):
        for j, name in enumerate(archetype_names):
            score = 0
            for feat, weight in archetypes[name].items():
                if feat in pn.columns:
                    score += weight * pn.loc[c, feat]
            score_matrix[i, j] = score
    
    # Si hay mas clusters que arquetipos, agregar columnas "Segmento X" con score 0
    if n_clusters > n_archetypes:
        extra = n_clusters - n_archetypes
        score_matrix = np.hstack([score_matrix, np.zeros((n_clusters, extra))])
        archetype_names += [f'Segmento {cluster_ids[k]}' for k in range(extra)]
    
    # Hungarian: maximizar score → minimizar -score
    row_idx, col_idx = linear_sum_assignment(-score_matrix)
    
    names = {}
    print("Scoring de asignacion (Hungarian):")
    for r, c_idx in zip(row_idx, col_idx):
        cluster_id = cluster_ids[r]
        name = archetype_names[c_idx]
        score = score_matrix[r, c_idx]
        names[cluster_id] = name
        print(f"  Cluster {cluster_id} → {name} (score={score:.2f})")
    
    return names

CLUSTER_NAMES = assign_cluster_names_hungarian(profiles_norm)
df_clust['cluster_name'] = df_clust['cluster'].map(CLUSTER_NAMES)

print(f"\nDistribucion:")
for c, name in sorted(CLUSTER_NAMES.items()):
    n = (df_clust['cluster'] == c).sum()
    print(f"  Cluster {c} = {name} ({n:,} clientes-t0)")

## 4. GMM Alternativo

In [ ]:
# ── GMM: comparar BIC/AIC y asignaciones ──────────────────────
gmm_results = []
for k in range(2, 9):
    gmm = GaussianMixture(n_components=k, random_state=42, n_init=3)
    gmm.fit(X_train_scaled)
    gmm_results.append({'k': k, 'BIC': gmm.bic(X_train_scaled), 'AIC': gmm.aic(X_train_scaled)})

gmm_df = pd.DataFrame(gmm_results)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(gmm_df['k'], gmm_df['BIC'], 'bo-', label='BIC')
ax.plot(gmm_df['k'], gmm_df['AIC'], 'rs-', label='AIC')
ax.set_xlabel('K')
ax.set_ylabel('Score')
ax.set_title('GMM: BIC y AIC por K (menor = mejor)')
ax.legend()
plt.tight_layout()
plt.show()

# Entrenar GMM con K_OPT
gmm_final = GaussianMixture(n_components=K_OPT, random_state=42, n_init=3)
gmm_final.fit(X_train_scaled)
labels_gmm = gmm_final.predict(X_scaled)

# Comparar KMeans vs GMM
ari = adjusted_rand_score(df_clust['cluster'], labels_gmm)
print(f"\nGMM K={K_OPT}: BIC={gmm_final.bic(X_train_scaled):,.0f}")
print(f"Adjusted Rand Index (KMeans vs GMM): {ari:.4f}")
print(f"  (1.0 = identicos, 0.0 = random)")
print(f"\nCrosstab KMeans vs GMM:")
print(pd.crosstab(df_clust['cluster'], labels_gmm, rownames=['KMeans'], colnames=['GMM']))

## 5. Visualización

In [ ]:
# ── PCA 2D scatter ───────────────────────────────────────────────
pca = PCA(n_components=2, random_state=42)
pca.fit(X_train_scaled)
X_pca = pca.transform(X_scaled)

# Centroides en PCA space
centroids_pca = pca.transform(kmeans.cluster_centers_)

fig, ax = plt.subplots(figsize=(10, 8))
colors = plt.cm.Set2(np.linspace(0, 1, K_OPT))

for c in range(K_OPT):
    mask = df_clust['cluster'].values == c
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], c=[colors[c]], alpha=0.3, s=15,
              label=f"{CLUSTER_NAMES[c]} (n={(mask).sum():,})")
    ax.scatter(centroids_pca[c, 0], centroids_pca[c, 1], c=[colors[c]],
              marker='X', s=200, edgecolor='black', linewidth=1.5)

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
ax.set_title(f'Clusters en PCA 2D (K={K_OPT})')
ax.legend(loc='best', fontsize=9)
plt.tight_layout()
plt.show()

print(f"Varianza explicada: PC1={pca.explained_variance_ratio_[0]*100:.1f}%, PC2={pca.explained_variance_ratio_[1]*100:.1f}%")

In [ ]:
# ── Radar chart por cluster ───────────────────────────────────────
from matplotlib.patches import FancyBboxPatch

categories = [f.replace('_', '\n') for f in CLUSTER_FEATURES]
N = len(categories)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]  # cerrar

fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))

for c in range(K_OPT):
    values = profiles_norm.loc[c].values.tolist()
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2, color=colors[c], label=CLUSTER_NAMES[c])
    ax.fill(angles, values, alpha=0.1, color=colors[c])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=8)
ax.set_title(f'Perfiles de Clusters (normalizado 0-1)', y=1.1, fontsize=13)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ── Distribucion de clusters por t0 (estabilidad temporal) ────────
ct_t0 = pd.crosstab(df_clust['t0'].dt.strftime('%Y-%m'), df_clust['cluster_name'], normalize='index') * 100

fig, ax = plt.subplots(figsize=(12, 6))
ct_t0.plot(kind='bar', stacked=True, ax=ax, colormap='Set2', width=0.7)
ax.set_ylabel('% del total')
ax.set_title('Distribucion de Clusters por t0 (centroides fijos)')
ax.legend(title='Cluster', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

print(ct_t0.round(1).to_string())

In [ ]:
# ── Heatmap: cluster x target y ─────────────────────────────────
ct_target = pd.crosstab(df_clust['cluster_name'], df_clust['y'], normalize='index') * 100
ct_target.columns = ['y=0 (No canje)', 'y=1 (Activacion)', 'y=2 (Recurrencia)']

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(ct_target, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax)
ax.set_title('% de cada Target por Cluster')
ax.set_ylabel('Cluster')
plt.tight_layout()
plt.show()

print("Observacion: Clusters con alto redeem_rate deberian tener mas y=1/y=2")

## 6. Tracking Temporal

In [ ]:
# ── Estabilidad y transiciones de clusters ─────────────────────
# Ordenar por cliente y t0
df_temp = df_clust[['cust_id', 't0', 'cluster', 'cluster_name']].sort_values(['cust_id', 't0'])

# Cluster mas frecuente por cliente
mode_cluster = df_temp.groupby('cust_id')['cluster'].agg(lambda x: x.mode()[0])

# Stability: % de t0s donde cluster == mode
df_temp = df_temp.merge(mode_cluster.rename('mode_cluster'), on='cust_id')
df_temp['same_as_mode'] = (df_temp['cluster'] == df_temp['mode_cluster']).astype(int)

stability = df_temp.groupby('cust_id')['same_as_mode'].mean()

# Transitions: conteo de cambios
df_temp['prev_cluster'] = df_temp.groupby('cust_id')['cluster'].shift(1)
df_temp['changed'] = (df_temp['cluster'] != df_temp['prev_cluster']) & df_temp['prev_cluster'].notna()
transitions = df_temp.groupby('cust_id')['changed'].sum()

print(f"Cluster Stability (% t0s en cluster modal):")
print(f"  mean={stability.mean():.2%}, median={stability.median():.2%}")
print(f"  100% estable: {(stability == 1.0).sum()} clientes ({(stability == 1.0).mean():.1%})")
print(f"\nTransiciones:")
print(f"  mean={transitions.mean():.2f}, max={transitions.max():.0f}")
print(f"  0 cambios: {(transitions == 0).sum()} clientes ({(transitions == 0).mean():.1%})")

In [ ]:
# ── Heatmap de migracion entre clusters (t0_n → t0_n+1) ─────────
df_mig = df_temp.dropna(subset=['prev_cluster']).copy()
df_mig['prev_cluster'] = df_mig['prev_cluster'].astype(int)

# Nombres para prev y current
df_mig['from_name'] = df_mig['prev_cluster'].map(CLUSTER_NAMES)
df_mig['to_name'] = df_mig['cluster'].map(CLUSTER_NAMES)

migration = pd.crosstab(df_mig['from_name'], df_mig['to_name'], normalize='index') * 100

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(migration, annot=True, fmt='.1f', cmap='Blues', ax=ax)
ax.set_title('Migracion entre Clusters (% fila = desde)')
ax.set_ylabel('Cluster origen (t0_n)')
ax.set_xlabel('Cluster destino (t0_n+1)')
plt.tight_layout()
plt.show()

# Diagonal = retencion
print("Retencion por cluster (diagonal):")
for name in migration.index:
    if name in migration.columns:
        print(f"  {name}: {migration.loc[name, name]:.1f}%")

## 7. Resumen

In [ ]:
# ── Tabla resumen con acciones sugeridas ───────────────────────
ACCIONES = {
    'Dormidos': 'Oferta directa / reactivacion',
    'Exploradores': 'Educar / onboarding de canjes',
    'Cazadores de Canje': 'Descuento / incentivo personalizado',
    'Heavy Users': 'Experiencias premium / exclusivas',
    'Digitales': 'Canal digital / app-first',
    'En Riesgo': 'Urgencia / retencion inmediata',
}

# Silhouette calculado solo en train (i.i.d., sin data leakage)
sil_final = silhouette_score(X_train_scaled, df_clust.loc[mask_train, 'cluster'].values)

print("="*90)
print(f"RESUMEN SEGMENTACION CONDUCTUAL — K={K_OPT}, Silhouette={sil_final:.4f}")
print("="*90)
print(f"\n{'Cluster':<5} {'Nombre':<22} {'N':<8} {'%':<8} {'Accion sugerida'}")
print("-"*90)

for c in sorted(CLUSTER_NAMES.keys()):
    name = CLUSTER_NAMES[c]
    n = (df_clust['cluster'] == c).sum()
    pct = n / len(df_clust) * 100
    accion = ACCIONES.get(name, '-')
    print(f"{c:<5} {name:<22} {n:<8,} {pct:<8.1f} {accion}")

print(f"\nEstabilidad temporal: {stability.mean():.1%} promedio")
print(f"Clientes 100% estables: {(stability == 1.0).mean():.1%}")
print(f"\nNota: Con datos mock (1000 clientes), los perfiles son orientativos.")
print(f"En produccion (500K clientes) los clusters seran mas diferenciados.")

---
## Next Steps

- Los clusters se integran al **Decision Engine** (Fase 2e) para personalizar acciones
- Los centroides fijos se guardan y aplican mensualmente en producción
- **Fase 2d**: Incrementalidad (PSM + Uplift)
- **Fase 2e**: Decision Engine (combina modelo, clusters, incrementalidad)